# Phase 2 — Scope & Dependency 60-Case Source Audit V3

**Purpose:** Continue the previous 90-case audit without changing the 30 Schedule decisions. Assess the remaining 30 Scope and 30 Dependency labels. **No paid API/GPU.**

**Source rules from teammate notebook:** 14-day future window `(checkpoint, checkpoint + 14 days]`; positive when at least **3 distinct affected issues**. Scope: eligible structural events, excluding OSSRH-I01 issue reclassifications on 2021-05-05 from `Task` to `Publishing Support`. Dependency: consolidated blocking-link `ADD` events, excluding specified 2020-09-13 bulk additions for JBIDE-I01, ENTESB-I01, BXMSDOC-I01 and 15 further JBIDE migration events.

**Evidence tiers:** `SOURCE_VERIFIED` requires independent raw Jira event identity + instance membership + eligibility + time; `PROCESSED_EVENT_AGREEMENT` compares with upstream event exports but is **not** independent source verification; `UNRESOLVED` means no defensible conclusion. This notebook never promotes processed-event agreement to ground truth.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import pandas as pd, numpy as np, json, glob
BASE=Path('/content/drive/MyDrive')
ROOT=BASE/'GC1'/'Temporal_snapshots'
OUT=BASE/'02_Risk_Prediction'/'Label_Validation'/'Results'/'Source_Reconstruction_V3_Scope_Dependency'
OUT.mkdir(parents=True,exist_ok=True)
print('Input root:',ROOT,'\nOutput:',OUT)

Mounted at /content/drive
Input root: /content/drive/MyDrive/GC1/Temporal_snapshots 
Output: /content/drive/MyDrive/02_Risk_Prediction/Label_Validation/Results/Source_Reconstruction_V3_Scope_Dependency


## 1. Locate the prior 90-case audit

This notebook requires `source_reconstructed_90_labels.csv` from V2. The `PRIOR_FILE` variable specifies the input location.

In [ ]:
matches=list(BASE.glob('02_Risk_Prediction/**/source_reconstructed_90_labels.csv'))
PRIOR_FILE=matches[0] if len(matches)==1 else None
print('Candidates:',[str(x) for x in matches])
if PRIOR_FILE is None:
    raise FileNotFoundError('Set PRIOR_FILE to the exact V2 source_reconstructed_90_labels.csv path')
prior=pd.read_csv(PRIOR_FILE)
required={'risk_type','instance_id','week_number','checkpoint_time','saved_label','saved_future_count','decision'}
assert required.issubset(prior.columns),required-set(prior.columns)
assert len(prior)==90 and prior['risk_type'].value_counts().to_dict()=={'schedule':30,'scope':30,'dependency':30}
assert prior.duplicated(['risk_type','instance_id','week_number','checkpoint_time']).sum()==0
remaining=prior[prior.risk_type.isin(['scope','dependency'])].copy()
remaining['checkpoint_dt']=pd.to_datetime(remaining.checkpoint_time,utc=True,format='mixed')
remaining['horizon_end']=remaining.checkpoint_dt+pd.Timedelta(days=14)
print('Remaining cases:',len(remaining))

Candidates: ['/content/drive/MyDrive/02_Risk_Prediction/Label_Validation/Results/Source_Reconstruction_V2/source_reconstructed_90_labels.csv']
Remaining cases: 60


## 2. Inspect available evidence exports

The scope structural-event parquet is expected from Notebook 1. The final dependency-ADD history might not have been exported. We **do not** substitute active dependency snapshots for new link events.

In [ ]:
for pat in ['*scope*events*.parquet','*depend*add*.parquet','*depend*histor*.parquet','*link*event*.parquet']:
    print(pat, [p.name for p in ROOT.glob(pat)])
SCOPE_FILE=ROOT/'historical_structural_scope_events.parquet'
DEP_FILE=None  # Only set if you have an export of final_eligible_adds from Notebook 1
if SCOPE_FILE.exists():
    scope=pd.read_parquet(SCOPE_FILE)
    print('Scope columns:',scope.columns.tolist(),'rows:',len(scope))
else:
    scope=None
    print('Scope export unavailable; scope cases remain unresolved')

*scope*events*.parquet ['historical_structural_scope_events.parquet']
*depend*add*.parquet []
*depend*histor*.parquet []
*link*event*.parquet []
Scope columns: ['canonical_id', 'entity_type', 'instance_id', 'label', 'name', 'payload_json', 'project_key', 'uid', 'created_at', 'observed_at', 'temporal_basis', 'temporal_status', 'chunk_id', 'source_id', 'source_timestamp', 'source_type', 'evidence_text', 'event_time', 'valid_from', 'valid_to', 'relation_type', 'event_type', 'jira_field', 'from_value', 'to_value', 'has_observed_at', 'event_dt', 'old_normalized', 'new_normalized', 'has_actual_change', 'has_both_values', 'structural_class', 'affected_issue', 'event_week', 'suspected_admin_reclassification'] rows: 938


## 3. Recompute Scope counts from the processed event export

This checks **upstream export-to-label consistency**. It does not establish independent original-Jira accuracy.

In [ ]:
def ts(s):return pd.to_datetime(s,utc=True,format='mixed',errors='coerce')
results=[]; evidence=[]
if scope is not None:
    need={'instance_id','event_dt','affected_issue','structural_class'}
    assert need.issubset(scope.columns),need-set(scope.columns)
    scope['event_dt']=ts(scope.event_dt)
    assert scope.event_dt.notna().all()
    assert 'suspected_admin_reclassification' not in scope or not scope['suspected_admin_reclassification'].fillna(False).astype(bool).any()
    if 'uid' in scope: assert scope['uid'].is_unique
for _,r in remaining.iterrows():
    result=dict(r.drop(labels=['checkpoint_dt','horizon_end']).to_dict())
    result.update({'recomputed_count':None,'recomputed_label':None,'processed_event_agreement':None,'source_verification':'UNRESOLVED','evidence_tier':'UNRESOLVED','notes':''})
    if r.risk_type=='scope' and scope is not None:
        x=scope[(scope.instance_id.astype(str)==str(r.instance_id)) & (scope.event_dt>r.checkpoint_dt) & (scope.event_dt<=r.horizon_end)]
        cnt=x.affected_issue.dropna().astype(str).nunique()
        result.update(recomputed_count=int(cnt),recomputed_label=int(cnt>=3),processed_event_agreement=bool(cnt==int(r.saved_future_count)),evidence_tier='PROCESSED_EVENT_AGREEMENT',notes='Derived from teammate final eligible structural-event export, not independently from raw Jira.')
        for _,e in x.iterrows():
            evidence.append({'risk_type':'scope','instance_id':r.instance_id,'week_number':r.week_number,'checkpoint_time':r.checkpoint_time,'event_uid':e.get('uid'),'event_dt':str(e.event_dt),'affected_issue':e.affected_issue,'structural_class':e.structural_class})
    elif r.risk_type=='dependency':
        result['notes']='No final eligible dependency ADD export configured. Cannot infer link additions from active dependency snapshots.'
    else:result['notes']='Scope eligible event export unavailable.'
    results.append(result)
audit=pd.DataFrame(results)
print(pd.crosstab(audit.risk_type,audit.evidence_tier))
display(audit[['risk_type','instance_id','week_number','saved_label','saved_future_count','recomputed_count','processed_event_agreement','evidence_tier']].head(12))

evidence_tier  PROCESSED_EVENT_AGREEMENT  UNRESOLVED
risk_type                                           
dependency                             0          30
scope                                 30           0


,risk_type,instance_id,week_number,saved_label,saved_future_count,recomputed_count,processed_event_agreement,evidence_tier
0,scope,SPARK-I01,1,0,1,1.0,True,PROCESSED_EVENT_AGREEMENT
1,scope,JRASERVER-I01,1,0,0,0.0,True,PROCESSED_EVENT_AGREEMENT
2,scope,QTCREATORBUG-I01,1,0,1,1.0,True,PROCESSED_EVENT_AGREEMENT
3,scope,MC-I01,1,0,0,0.0,True,PROCESSED_EVENT_AGREEMENT
4,scope,FLEX-I01,1,0,1,1.0,True,PROCESSED_EVENT_AGREEMENT
5,scope,HBASE-I01,1,0,2,2.0,True,PROCESSED_EVENT_AGREEMENT
6,scope,SPR-I01,1,0,0,0.0,True,PROCESSED_EVENT_AGREEMENT
7,scope,RUBY-I01,1,0,2,2.0,True,PROCESSED_EVENT_AGREEMENT
8,scope,JBEAP-I01,2,0,1,1.0,True,PROCESSED_EVENT_AGREEMENT
9,scope,AEROGEAR-I01,1,0,0,0.0,True,PROCESSED_EVENT_AGREEMENT


## 4. Optional: dependency final eligible ADD export

Where the upstream workflow exports `final_eligible_adds` **without modifying the data**, save it as `final_eligible_dependency_adds.parquet` in `GC1/Temporal_snapshots`. Required columns: `instance_id`, `event_dt`, `downstream_issue`, and ideally `event_uid`, `upstream_issue`. The export should already reflect link-direction consolidation and bulk/migration exclusions. Recomputing from it tests pipeline consistency, not independent Jira source truth.

In [1]:
DEP_FILE=ROOT/'final_eligible_dependency_adds.parquet'
if DEP_FILE.exists():
    dep=pd.read_parquet(DEP_FILE)
    req={'instance_id','event_dt','downstream_issue'}
    assert req.issubset(dep.columns),req-set(dep.columns)
    dep['event_dt']=ts(dep.event_dt)
    assert dep.event_dt.notna().all()
    if 'action' in dep:assert dep.action.eq('ADD').all()
    for flag in ['exclude_from_escalation','suspected_bulk_update','suspected_jbide_migration']:
        if flag in dep:assert not dep[flag].fillna(False).astype(bool).any(),f'Excluded rows in {flag}'
    for i,r in remaining[remaining.risk_type=='dependency'].iterrows():
        x=dep[(dep.instance_id.astype(str)==str(r.instance_id)) & (dep.event_dt>r.checkpoint_dt) & (dep.event_dt<=r.horizon_end)]
        cnt=x.downstream_issue.dropna().astype(str).nunique()
        idx=audit.index[(audit.risk_type=='dependency')&(audit.instance_id==r.instance_id)&(audit.week_number==r.week_number)]
        assert len(idx)==1
        j=idx[0]
        audit.loc[j,['recomputed_count','recomputed_label','processed_event_agreement','evidence_tier','notes']]=[int(cnt),int(cnt>=3),bool(cnt==int(r.saved_future_count)),'PROCESSED_EVENT_AGREEMENT','Derived from teammate final eligible dependency ADD export, not independently raw Jira.']
        for _,e in x.iterrows():evidence.append({'risk_type':'dependency','instance_id':r.instance_id,'week_number':r.week_number,'checkpoint_time':r.checkpoint_time,'event_uid':e.get('event_uid'),'event_dt':str(e.event_dt),'affected_issue':e.downstream_issue,'structural_class':'BLOCKING_LINK_ADD'})
    print('Dependency export processed:',len(dep))
else:print('Dependency eligible ADD export not found. 30 dependency cases remain unresolved.')

NameError: name 'ROOT' is not defined

## 5. Save results and interpret carefully

Even 60/60 matches against the upstream exports would not prove independent ground-truth validity. Exact raw Jira issue/link changelog reconstruction, issue-to-instance membership, and deduplication must be audited separately. A mismatch is a genuine pipeline discrepancy to investigate, not permission to edit ground truth.

In [ ]:
audit.to_csv(OUT/'scope_dependency_60_case_audit.csv',index=False)
pd.DataFrame(evidence,columns=['risk_type','instance_id','week_number','checkpoint_time','event_uid','event_dt','affected_issue','structural_class']).to_csv(OUT/'processed_event_evidence.csv',index=False)
summary={'cases':len(audit),'scope_cases':30,'dependency_cases':30,'processed_event_agreement_true':int(audit.processed_event_agreement.eq(True).sum()),'processed_event_agreement_false':int(audit.processed_event_agreement.eq(False).sum()),'not_tested':int(audit.processed_event_agreement.isna().sum()),'independently_source_verified':0,'source_accuracy':'NOT_ESTABLISHED','limitations':['Upstream event-export agreement is not independent original Jira verification','Dependency eligible ADD export optional and may be missing','Exact issue-to-instance membership and original raw Jira event eligibility not independently validated','Never interpret untested as negative or incorrect']}
(OUT/'scope_dependency_source_audit_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2));print('Saved to',OUT)

{
  "cases": 60,
  "scope_cases": 30,
  "dependency_cases": 30,
  "processed_event_agreement_true": 30,
  "processed_event_agreement_false": 0,
  "not_tested": 30,
  "independently_source_verified": 0,
  "source_accuracy": "NOT_ESTABLISHED",
  "limitations": [
    "Upstream event-export agreement is not independent original Jira verification",
    "Dependency eligible ADD export optional and may be missing",
    "Exact issue-to-instance membership and original raw Jira event eligibility not independently validated",
    "Never interpret untested as negative or incorrect"
  ]
}
Saved to /content/drive/MyDrive/02_Risk_Prediction/Label_Validation/Results/Source_Reconstruction_V3_Scope_Dependency
